# Training ML-AMC (AMCFeatureNet) on RadioML 2018.01A

This notebook provides an automated training and evaluation pipeline for **Automatic Modulation Classification (ML-AMC)** directly on the **[RadioML 2018.01A Dataset](https://www.kaggle.com/datasets/pinxau1000/radioml2018/data)** (`GOLD_XYZ_OSHF.hdf5`).

## Pipeline Highlights
- **DSP Feature Extraction**: Vectorized extraction of 40 physical and dual-order Costas loop tracking features using `gr_playground.dsp.amc.features`.
- **Feature Caching**: Automatic disk caching (`radioml2018_features_cached.npz`) to allow rapid re-runs and hyperparameter exploration without re-extracting.
- **80:20 Train / Validation Split**: Stratified sampling across all 24 modulation formats.
- **PyTorch AMCFeatureNet**: Deep classifier with input batch normalization, dropout, and AdamW optimization with Cosine Annealing learning rate schedule.
- **Visualizations**:
  - Training and Validation Loss & Accuracy curves with respect to epoch.
  - Normalized 24-class Confusion Matrix heatmap.
  - Accuracy vs Signal-to-Noise Ratio (SNR) curves (-20 dB to +30 dB).
- **ONNX Export**: Direct export to portable `radioml2018_amc_model.onnx` with latency benchmark.

In [ ]:
!python --version

In [ ]:
# 1. Update apt repositories and install GNU Radio system package
print("Updating...")
!apt-get update -qq
print("Installing...")
!apt-get install -qq -y gnuradio > /dev/null

import sys
sys.path.append("/usr/lib/python3/dist-packages")

In [ ]:
# 1. Environment Setup & Dependency Verification
import os
import sys
import subprocess

print("Python version:", sys.version)

# Install required packages if missing in Kaggle environment
required_pkgs = ["h5py", "onnx", "onnxruntime", "seaborn", "tqdm", "scikit-learn", "pandas"]
for pkg in required_pkgs:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# Setup gr-playground repository from GitHub or local path
try:
    import gr_playground
    print("gr_playground already installed!")
except ImportError:
    print("Setting up gr_playground from GitHub...")
    if not os.path.exists("gr-playground"):
        subprocess.check_call(["git", "clone", "https://github.com/nitrojacob/gr-playground.git"])
    repo_path = os.path.abspath("gr-playground")
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)
    import gr_playground
    print("gr_playground successfully loaded!")

import numpy as np
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
from tqdm.auto import tqdm

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

# Configure compute device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")
if device.type == "cuda":
    print(f"Device name: {torch.cuda.get_device_name(0)}")
else:
    num_threads = min(os.cpu_count() or 4, 8)
    torch.set_num_threads(num_threads)
    print(f"Configured PyTorch CPU threads: {num_threads}")

## 1. Locate RadioML 2018 Dataset

RadioML 2018.01A (`GOLD_XYZ_OSHF.hdf5` or `GOLD_XYZ_OSC.0001_1024.hdf5`) contains 2,555,904 frames of 1024 complex IQ samples across 24 modulation schemes and 26 SNR steps (-20 dB to +30 dB in 2 dB steps).

In [ ]:
RADIOML_24_CLASSES = [
    'OOK', '4ASK', '8ASK', 'BPSK', 'QPSK', '8PSK', '16PSK', '32PSK',
    '16APSK', '32APSK', '64APSK', '128APSK', '16QAM', '32QAM', '64QAM',
    '128QAM', '256QAM', 'AM-SSB-WC', 'AM-SSB-SC', 'AM-DSB-WC', 'AM-DSB-SC',
    'FM', 'GFSK', 'CPFSK'
]

RADIOML_TO_PARENT_MAP = {
    'AM-SSB-WC': 'AM', 'AM-SSB-SC': 'AM', 'AM-DSB-WC': 'AM', 'AM-DSB-SC': 'AM',
    'FM': 'FM', 'BPSK': 'BPSK', 'GFSK': 'GFSK', 'CPFSK': 'GFSK',
    'QPSK': 'QPSK', '8PSK': '8PSK', '16PSK': '8PSK', '32PSK': '8PSK',
    '16QAM': '16QAM', '32QAM': '16QAM', '64QAM': '64QAM', '128QAM': '64QAM',
    '256QAM': '256QAM', 'OOK': 'ASK', '4ASK': 'ASK', '8ASK': 'ASK',
    '16APSK': '16APSK', '32APSK': '16APSK', '64APSK': '32APSK', '128APSK': '32APSK'
}

def locate_radioml_dataset():
    """Search common Kaggle and local paths for RadioML 2018 HDF5 file."""
    candidate_paths = [
        "/kaggle/input/datasets/pinxau1000/radioml2018/GOLD_XYZ_OSHF.hdf5",
        "/kaggle/input/datasets/pinxau1000/radioml2018/GOLD_XYZ_OSC.0001_1024.hdf5",
        "/kaggle/input/radioml2018/GOLD_XYZ_OSHF.hdf5",
        "/kaggle/input/radioml2018/GOLD_XYZ_OSC.0001_1024.hdf5",
        "/kaggle/input/radioml2018/data/GOLD_XYZ_OSHF.hdf5",
        "/kaggle/input/radioml2018/data/GOLD_XYZ_OSC.0001_1024.hdf5",
        "/kaggle/input/radioml2018.01a/GOLD_XYZ_OSHF.hdf5",
        "/kaggle/input/radioml2018.01a/GOLD_XYZ_OSC.0001_1024.hdf5",
        "/kaggle/input/radioml2018/radioml2018/GOLD_XYZ_OSHF.hdf5",
        "/kaggle/input/radioml2018/radioml2018/GOLD_XYZ_OSC.0001_1024.hdf5",
        "GOLD_XYZ_OSHF.hdf5",
        "GOLD_XYZ_OSC.0001_1024.hdf5",
        "../data/GOLD_XYZ_OSHF.hdf5",
        "data/GOLD_XYZ_OSHF.hdf5",
    ]
    for path in candidate_paths:
        if os.path.exists(path):
            return path
    return None

dataset_path = locate_radioml_dataset()
if dataset_path:
    print(f"Found RadioML 2018 dataset at: {dataset_path}")
    with h5py.File(dataset_path, "r") as f:
        print("Dataset HDF5 keys:", list(f.keys()))
        print("X shape:", f["X"].shape, f["X"].dtype)
        if "Y" in f:
            print("Y shape:", f["Y"].shape)
        if "Z" in f:
            print("Z shape:", f["Z"].shape)
else:
    print("RadioML 2018 dataset file not found in default paths.")
    print("Please attach dataset 'pinxau1000/radioml2018' to your Kaggle Notebook inputs.")

## 2. DSP Feature Extraction & Data Preparation

Extract 40 physical and dual-order Costas loop tracking features from complex IQ frames using `extract_frame_features`:
- **Statistical Cumulants**: C20, C21, C40, C41, C42, C60, C63
- **Spectral & Envelope Moments**: Variance, skewness, kurtosis, peak-to-average power ratio (PAPR)
- **Costas Loop Features**: 2nd-order and 4th-order Costas loop phase error variance and lock indicators (isolating PSK from QAM/AM)

Extracted features are cached to `radioml2018_features_cached.npz` so re-runs do not recompute features.

In [ ]:
import time
from concurrent.futures import ProcessPoolExecutor
from gr_playground.dsp.amc.features import extract_frame_features

CACHE_FILE = "radioml2018_features_cached.npz"
SAMPLES_PER_CLASS_SNR = 100  # Number of frames per (class, snr). Total samples = 24 * 26 * 100 = 62,400.
SNR_LEVELS = list(range(-20, 32, 2))  # 26 SNR steps (-20 dB to +30 dB)
FRAMES_PER_BLOCK = 4096

def _extract_batch_worker(args):
    """Worker function to extract 40 features for a batch of frames."""
    raw_iq_batch, class_idx, snr_val = args
    n = len(raw_iq_batch)
    batch_feats = np.zeros((n, 40), dtype=np.float32)
    # Convert interleaved IQ [N, 1024, 2] to complex64
    complex_iq = (raw_iq_batch[:, :, 0] + 1j * raw_iq_batch[:, :, 1]).astype(np.complex64)
    for i in range(n):
        batch_feats[i] = extract_frame_features(complex_iq[i])
    batch_labels = np.full(n, class_idx, dtype=np.int64)
    batch_snrs = np.full(n, snr_val, dtype=np.float32)
    return batch_feats, batch_labels, batch_snrs

def load_or_extract_features(hdf5_path, samples_per_class_snr=100, cache_path=CACHE_FILE):
    """Load cached features or extract from HDF5 dataset in parallel."""
    if os.path.exists(cache_path):
        print(f"Loading cached features from {cache_path}...")
        data = np.load(cache_path)
        X = data['X']
        y = data['y']
        snrs = data['snrs']
        print(f"Loaded {len(X):,} samples with feature shape: {X.shape}")
        return X, y, snrs

    if not hdf5_path or not os.path.exists(hdf5_path):
        raise FileNotFoundError(f"HDF5 dataset not found: {hdf5_path}")

    print(f"Extracting features from {hdf5_path} ({samples_per_class_snr} samples per class/SNR)...")
    t0 = time.time()
    tasks = []

    with h5py.File(hdf5_path, 'r', swmr=True) as f:
        X_ds = f['X']
        for s_idx, snr_val in enumerate(SNR_LEVELS):
            for c_idx, c_name in enumerate(RADIOML_24_CLASSES):
                offset = (c_idx * 26 + s_idx) * FRAMES_PER_BLOCK
                count = min(samples_per_class_snr, FRAMES_PER_BLOCK)
                raw_batch = X_ds[offset : offset + count]
                tasks.append((raw_batch, c_idx, snr_val))

    print(f"Total batches to extract: {len(tasks)}. Running parallel extraction across CPU cores...")
    max_workers = min(os.cpu_count() or 4, 8)
    X_list, y_list, snr_list = [], [], []

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        for b_feats, b_labels, b_snrs in executor.map(_extract_batch_worker, tasks):
            X_list.append(b_feats)
            y_list.append(b_labels)
            snr_list.append(b_snrs)

    X = np.vstack(X_list).astype(np.float32)
    y = np.concatenate(y_list).astype(np.int64)
    snrs = np.concatenate(snr_list).astype(np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=10.0, neginf=-10.0)

    elapsed = time.time() - t0
    print(f"Extraction completed in {elapsed:.2f}s ({len(X)/max(elapsed, 0.001):.1f} samples/sec).")
    print(f"Caching features to {cache_path}...")
    np.savez_compressed(cache_path, X=X, y=y, snrs=snrs)
    return X, y, snrs

if dataset_path:
    X_feats, y_labels, snr_tags = load_or_extract_features(
        dataset_path, 
        samples_per_class_snr=SAMPLES_PER_CLASS_SNR, 
        cache_path=CACHE_FILE
    )
    print(f"Feature matrix shape: {X_feats.shape}, Labels shape: {y_labels.shape}")

## 3. 80:20 Train / Validation Split

Split the dataset into an **80% training set** and a **20% validation set** using stratified sampling to ensure balanced class distributions.

In [ ]:
from sklearn.model_selection import train_test_split

# Perform 80:20 stratified split based on class labels
X_train, X_val, y_train, y_val, snr_train, snr_val = train_test_split(
    X_feats, y_labels, snr_tags,
    test_size=0.20,
    random_state=42,
    stratify=y_labels
)

print(f"Total Samples:        {len(X_feats):,}")
print(f"Training Set (80%):   {len(X_train):,} samples")
print(f"Validation Set (20%): {len(X_val):,} samples")

# Convert to PyTorch Tensors
train_tensor_x = torch.tensor(X_train, dtype=torch.float32)
train_tensor_y = torch.tensor(y_train, dtype=torch.long)
val_tensor_x = torch.tensor(X_val, dtype=torch.float32)
val_tensor_y = torch.tensor(y_val, dtype=torch.long)

BATCH_SIZE = 256
train_dataset = TensorDataset(train_tensor_x, train_tensor_y)
val_dataset = TensorDataset(val_tensor_x, val_tensor_y)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoader ready: Batch Size = {BATCH_SIZE}, Train Batches = {len(train_loader)}, Val Batches = {len(val_loader)}")

## 4. PyTorch AMCFeatureNet Model Architecture

`AMCFeatureNet` is a deep neural classifier designed for modulation classification from physical feature vectors:
- Input Batch Normalization layer (`BatchNorm1d(40)`) to standardize heterogeneous statistical features.
- Dense layers (256 -> 128 -> 64) with Batch Normalization, ReLU activations, and Dropout regularization.
- Output Linear projection to the 24 RadioML modulation classes.

In [ ]:
class AMCFeatureNet(nn.Module):
    """
    PyTorch Feature Classifier mapping 40 physical DSP features to modulation logits.
    """
    def __init__(self, input_dim=40, num_classes=24):
        super().__init__()
        self.input_bn = nn.BatchNorm1d(input_dim)
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(self.input_bn(x))

model = AMCFeatureNet(input_dim=40, num_classes=len(RADIOML_24_CLASSES)).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"Total Trainable Parameters: {total_params:,}")

## 5. Model Training & Epoch Metric Tracking

Train `AMCFeatureNet` using AdamW and a Cosine Annealing learning rate schedule. We track both training loss, training accuracy, validation loss, and validation accuracy for each epoch.

In [ ]:
EPOCHS = 60
LEARNING_RATE = 3e-3
WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

history = {
    'epoch': [],
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'lr': []
}

best_val_acc = 0.0
best_model_state = None

print(f"Starting training on {device} for {EPOCHS} epochs...")
start_time = time.time()

pbar = tqdm(range(1, EPOCHS + 1), desc="Training AMCFeatureNet")
for epoch in pbar:
    # Training Phase
    model.train()
    running_train_loss = 0.0
    correct_train = 0
    total_train = 0

    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * bx.size(0)
        preds = torch.argmax(logits, dim=1)
        correct_train += (preds == by).sum().item()
        total_train += bx.size(0)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    train_epoch_loss = running_train_loss / total_train
    train_epoch_acc = (correct_train / total_train) * 100.0

    # Validation Phase
    model.eval()
    running_val_loss = 0.0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            loss = criterion(logits, by)

            running_val_loss += loss.item() * bx.size(0)
            preds = torch.argmax(logits, dim=1)
            correct_val += (preds == by).sum().item()
            total_val += bx.size(0)

    val_epoch_loss = running_val_loss / total_val
    val_epoch_acc = (correct_val / total_val) * 100.0

    # Record history
    history['epoch'].append(epoch)
    history['train_loss'].append(train_epoch_loss)
    history['train_acc'].append(train_epoch_acc)
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)
    history['lr'].append(current_lr)

    if val_epoch_acc > best_val_acc:
        best_val_acc = val_epoch_acc
        best_model_state = model.state_dict().copy()

    pbar.set_postfix({
        'loss': f"{train_epoch_loss:.3f}",
        'acc': f"{train_epoch_acc:.1f}%",
        'val_loss': f"{val_epoch_loss:.3f}",
        'val_acc': f"{val_epoch_acc:.1f}%"
    })

    if epoch % 5 == 0 or epoch == 1 or epoch == EPOCHS:
        pbar.write(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | "
                   f"Train Loss: {train_epoch_loss:.4f} | Train Acc: {train_epoch_acc:5.2f}% | "
                   f"Val Loss: {val_epoch_loss:.4f} | Val Acc: {val_epoch_acc:5.2f}% | "
                   f"LR: {current_lr:.6f}")

total_duration = time.time() - start_time
print("=" * 70)
print(f"Training completed in {total_duration:.2f}s ({total_duration/EPOCHS:.2f}s/epoch).")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

# Load best checkpoint weights
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print("Loaded best model state based on validation accuracy.")

# Format Training Progress Report Table
progress_df = pd.DataFrame(history)
print("\n" + "=" * 35 + " TOP 5 BEST EPOCHS " + "=" * 35)
print(progress_df.sort_values(by="val_acc", ascending=False).head(5).to_string(index=False))

## 6. Visualization: Training & Validation Accuracy and Loss wrt Epoch

Visualize the learning curves showing both loss and accuracy for training and validation sets with respect to epoch.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

epochs_arr = history['epoch']
best_epoch_idx = int(np.argmax(history['val_acc']))
best_epoch = epochs_arr[best_epoch_idx]

# Subplot 1: Loss vs Epoch
axes[0].plot(epochs_arr, history['train_loss'], 'o-', color='#1f77b4', linewidth=2, label='Training Loss')
axes[0].plot(epochs_arr, history['val_loss'], 's-', color='#ff7f0e', linewidth=2, label='Validation Loss')
axes[0].axvline(best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best Val Epoch ({best_epoch})')
axes[0].set_title('Training and Validation Loss vs Epoch', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=11)
axes[0].legend(loc='upper right', frameon=True)
axes[0].grid(True, alpha=0.3)

# Subplot 2: Accuracy vs Epoch
axes[1].plot(epochs_arr, history['train_acc'], 'o-', color='#2ca02c', linewidth=2, label='Training Accuracy')
axes[1].plot(epochs_arr, history['val_acc'], 's-', color='#d62728', linewidth=2, label='Validation Accuracy')
axes[1].axvline(best_epoch, color='gray', linestyle='--', alpha=0.7, label=f'Best Val Acc ({best_val_acc:.2f}%)')
axes[1].scatter([best_epoch], [best_val_acc], color='#d62728', s=120, zorder=5)
axes[1].set_title('Training and Validation Accuracy vs Epoch', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Accuracy (%)', fontsize=11)
axes[1].set_ylim(0, 105)
axes[1].legend(loc='lower right', frameon=True)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("amc_training_validation_curves.png", dpi=300)
plt.show()
print("Saved plot to: amc_training_validation_curves.png")

## 7. Validation Evaluation & Confusion Matrix

Evaluate the best model checkpoint on the held-out validation set across all 24 classes:
- **Exact 24-Class Top-1 Accuracy**
- **Modulation Family Accuracy** (grouping PSK, QAM, APSK, ASK, AM, and FM/GFSK families)
- **Normalized Confusion Matrix Heatmap**

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
val_preds_list = []
val_targets_list = []

with torch.no_grad():
    for bx, by in val_loader:
        bx = bx.to(device)
        logits = model(bx)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        val_preds_list.extend(preds)
        val_targets_list.extend(by.numpy())

val_preds = np.array(val_preds_list)
val_targets = np.array(val_targets_list)

# 1. Exact 24-Class Accuracy
exact_val_acc = (val_preds == val_targets).mean() * 100.0

# 2. Family-Level Accuracy
family_correct = 0
for pred_idx, true_idx in zip(val_preds, val_targets):
    pred_cls = RADIOML_24_CLASSES[pred_idx]
    true_cls = RADIOML_24_CLASSES[true_idx]
    if RADIOML_TO_PARENT_MAP.get(pred_cls, pred_cls) == RADIOML_TO_PARENT_MAP.get(true_cls, true_cls):
        family_correct += 1
family_val_acc = (family_correct / len(val_targets)) * 100.0

print(f"Validation Exact 24-Class Accuracy: {exact_val_acc:.2f}%")
print(f"Validation Family-Level Accuracy:  {family_val_acc:.2f}%")
print("-" * 65)

# Classification Report
print(classification_report(val_targets, val_preds, target_names=RADIOML_24_CLASSES, digits=3))

# Confusion Matrix Heatmap
cm = confusion_matrix(val_targets, val_preds, labels=range(len(RADIOML_24_CLASSES)), normalize='true')

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=RADIOML_24_CLASSES,
    yticklabels=RADIOML_24_CLASSES,
    cbar=True
)
plt.title(f'RadioML 2018 24-Class Confusion Matrix (Overall Val Acc: {exact_val_acc:.1f}%)', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Class', fontsize=11)
plt.ylabel('Ground Truth Class', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("amc_confusion_matrix.png", dpi=300)
plt.show()

## 8. Accuracy vs Signal-to-Noise Ratio (SNR)

Measure how model classification accuracy scales across SNR regimes on the validation set, comparing exact 24-class accuracy and family-level accuracy from -20 dB to +30 dB.

In [ ]:
snrs_unique = np.sort(np.unique(snr_val))
exact_acc_by_snr = []
family_acc_by_snr = []

for s in snrs_unique:
    mask = (snr_val == s)
    sub_preds = val_preds[mask]
    sub_targets = val_targets[mask]
    if len(sub_targets) == 0:
        continue

    exact = (sub_preds == sub_targets).mean() * 100.0
    exact_acc_by_snr.append(exact)

    fam_correct = sum(
        RADIOML_TO_PARENT_MAP.get(RADIOML_24_CLASSES[p], RADIOML_24_CLASSES[p]) ==
        RADIOML_TO_PARENT_MAP.get(RADIOML_24_CLASSES[t], RADIOML_24_CLASSES[t])
        for p, t in zip(sub_preds, sub_targets)
    )
    family_acc_by_snr.append((fam_correct / len(sub_targets)) * 100.0)

plt.figure(figsize=(10, 6))
plt.plot(snrs_unique, family_acc_by_snr, 's-', color='#2ca02c', linewidth=2.5, label='Family-Level Accuracy')
plt.plot(snrs_unique, exact_acc_by_snr, 'o-', color='#1f77b4', linewidth=2.5, label='Exact 24-Class Accuracy')
plt.axhline(90, color='red', linestyle='--', alpha=0.7, label='90% Target')
plt.title('Validation Accuracy vs SNR on RadioML 2018.01A', fontsize=13, fontweight='bold')
plt.xlabel('Signal-to-Noise Ratio (SNR in dB)', fontsize=11)
plt.ylabel('Accuracy (%)', fontsize=11)
plt.ylim(0, 105)
plt.legend(loc='lower right', frameon=True)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("amc_accuracy_vs_snr.png", dpi=300)
plt.show()

## 9. Export Trained Model to ONNX

Export the trained PyTorch model to an ONNX runtime artifact (`radioml2018_amc_model.onnx`).
Verify the ONNX model using `onnxruntime` and measure average inference latency per frame.

In [ ]:
import onnx
import onnxruntime as ort

onnx_output_path = "radioml2018_amc_model.onnx"
model.eval()

# Move model to CPU for standard export
cpu_model = model.to("cpu")
dummy_input = torch.randn(1, 40, dtype=torch.float32)

torch.onnx.export(
    cpu_model,
    dummy_input,
    onnx_output_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}}
)
print(f"Successfully exported ONNX model artifact to: {onnx_output_path}")

# Verify ONNX model integrity
onnx_proto = onnx.load(onnx_output_path)
onnx.checker.check_model(onnx_proto)
print("ONNX model structure validated successfully.")

# Test inference session and benchmark execution latency
session = ort.InferenceSession(onnx_output_path, providers=["CPUExecutionProvider"])
test_feat = np.random.randn(1, 40).astype(np.float32)

# Warmup
for _ in range(20):
    _ = session.run(None, {"input": test_feat})

t0 = time.perf_counter()
N_RUNS = 200
for _ in range(N_RUNS):
    out = session.run(None, {"input": test_feat})
t1 = time.perf_counter()

avg_latency_ms = ((t1 - t0) / N_RUNS) * 1000.0
print(f"Average ONNX Inference Latency: {avg_latency_ms:.3f} ms / sample ({1000.0/avg_latency_ms:.1f} predictions/sec)")
print(f"Output logits shape: {out[0].shape}")

## 10. Summary & Kaggle Output Artifacts

Artifacts produced:
- `amc_training_validation_curves.png`: Training & validation loss and accuracy vs epoch.
- `amc_confusion_matrix.png`: Normalized 24-class confusion matrix on the validation set.
- `amc_accuracy_vs_snr.png`: Top-1 accuracy curve across SNR levels (-20 dB to +30 dB).
- `radioml2018_amc_model.onnx`: Standalone portable ONNX model file.
- `radioml2018_features_cached.npz`: Cached 40-dimensional feature arrays.

These artifacts are available in the Kaggle working directory (`/kaggle/working/`) and can be downloaded or used in downstream SDR pipelines.